# Calculatrice seq2seq

Dans ces travaux pratiques, un réseau récurrent apprend à calculer à partir de texte : il lit une équation écrite caractère par caractère, comme `12+34`, et écrit son résultat, `46`, caractère par caractère lui aussi. C'est un modèle *sequence to sequence* (seq2seq) : un encodeur résume la séquence d'entrée, un décodeur produit la séquence de sortie.

1. Encoder les équations.
2. Générer le jeu de données.
3. Construire l'encodeur-décodeur.
4. L'entraîner, puis l'évaluer.
5. La réussite par opération.
6. Faut-il inverser l'équation ?

Ce TP a besoin d'une carte graphique : vérifiez dans `Exécution > Modifier le type d'exécution` que l'accélérateur matériel est un GPU. Sur un processeur, ses deux entraînements prendraient plusieurs heures.

## Installation et import des bibliothèques

In [ ]:
!pip install -q lightning torchmetrics

In [ ]:
import operator
import pathlib
from collections.abc import Sequence

import lightning
import matplotlib.pyplot as plt
import numpy as np
import pandas
import sklearn.model_selection
import torch
import torchmetrics
from lightning.pytorch.callbacks import Callback, TQDMProgressBar
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.utilities.model_summary import ModelSummary
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

# Mêmes tirages aléatoires à chaque exécution : des résultats reproductibles
lightning.seed_everything(0)

## 1. Encoder les équations

Les équations et leurs résultats s'écrivent avec 16 caractères : les chiffres, le point décimal, les 4 opérations et l'espace, qui sert de *padding*. `encode` transforme une chaîne en tenseur d'indices de ce vocabulaire, `decode` fait l'inverse.

In [ ]:
operations = list("+*-/")
numbers = list("0123456789.")
padding = [" "]

index_to_char = numbers + operations + padding
char_to_index = {char: index for index, char in enumerate(index_to_char)}


def encode(characters: str) -> torch.Tensor:
  return torch.tensor([char_to_index[char] for char in characters])


def decode(indices: torch.Tensor) -> str:
  return "".join(index_to_char[int(index)] for index in indices)


print(f"Vocabulaire : {index_to_char}")
print(f'encode("3/4+2-5") = {encode("3/4+2-5")}')
print(f'decode(encode("3/4+2-5")) = {decode(encode("3/4+2-5"))}')

### Quelle entrée pour le réseau ?

Le réseau peut-il travailler directement sur la sortie de `encode`, des entiers de 0 à 15, ou faut-il une transformation de plus ? Laquelle ?

Votre réponse ici

#### Solution

Ces indices désignent des catégories, pas des quantités : que `+` ait l'indice 11 et `9` l'indice 9 ne veut pas dire que `+` vaut plus que `9`. Donnés tels quels, ils feraient croire au réseau à un ordre et à des distances qui n'existent pas. Il faut donc, au choix :

- un encodage *one-hot* : un vecteur de 16 valeurs, toutes nulles sauf un 1 à l'indice du caractère ;
- une couche `nn.Embedding`, qui apprend un vecteur par caractère, comme dans les slides. C'est ce que fera notre modèle.

## 2. Générer le jeu de données

`make_maths_problem` tire une opération au hasard et deux entiers de 0 à 999, jusqu'à obtenir le nombre d'équations demandé, toutes différentes :

- les divisions par 0 sont écartées ;
- les résultats sont tronqués à 6 caractères (`0.3333`) ;
- équations et résultats sont complétés par des espaces, pour faire tous 7 et 6 caractères ;
- l'équation est inversée par défaut (`invert=True`) : `12+34` devient `43+21`. La partie 6 dit pourquoi.

In [ ]:
def make_maths_problem(
  n_samples: int = 1000, n_digits: int = 3, invert: bool = True
) -> tuple[torch.Tensor, torch.Tensor]:
  equations = []
  results = []
  seen = set()

  math_operation = {
    "+": operator.add,
    "-": operator.sub,
    "*": operator.mul,
    "/": operator.truediv,
  }

  # Longueurs maximales de l'équation et du résultat
  max_equation_len = 2 * n_digits + 1
  max_result_len = 2 * n_digits

  while len(equations) < n_samples:
    operation = np.random.choice(operations)
    left, right = np.random.randint(10**n_digits, size=2)
    equation = f"{left}{operation}{right}"
    # Une équation déjà tirée, ou une division par 0 : on recommence
    if equation in seen or (operation == "/" and right == 0):
      continue
    seen.add(equation)

    # Un résultat comme 0.3333333… est tronqué
    result = str(math_operation[operation](left, right))[:max_result_len]
    # Des espaces pour que toutes les séquences aient la même longueur
    padded_equation = equation.ljust(max_equation_len)
    padded_result = result.ljust(max_result_len)
    if invert:
      padded_equation = padded_equation[::-1]

    equations.append(padded_equation)
    results.append(padded_result)

  X = torch.stack([encode(equation) for equation in equations])
  y = torch.stack([encode(result) for result in results])
  return X, y


X, y = make_maths_problem(10)
print(f"Formes de X et y : {tuple(X.shape)}, {tuple(y.shape)}")
for encoded_equation, encoded_result in zip(X, y):
  # L'équation est inversée : on la remet dans l'ordre pour l'afficher
  print(f"{decode(encoded_equation.flip(0))} = {decode(encoded_result)}")

Le jeu de données : 100 000 équations, dont 80 % pour l'entraînement et 20 % pour le test.

In [ ]:
X, y = make_maths_problem(100_000)
X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(
  X, y, train_size=0.8, random_state=0
)
print(tuple(X_train.shape), tuple(X_test.shape))

## 3. Construire l'encodeur-décodeur

Écrivez la classe `Seq2Seq`, un `nn.Module` :

- `__init__(hidden_size=512)` crée ses couches :
  - **l'encodeur** : une couche `nn.Embedding` de 16 dimensions, puis un `nn.GRU` de deux couches empilées de `hidden_size` unités ;
  - **le décodeur** : un `nn.GRU` d'une couche de `hidden_size` unités, puis une couche `nn.Linear` qui donne un score (un logit) à chacun des 16 caractères ;
- `forward(equations)` reçoit des équations encodées, de forme `(batch, 7)` (`max_equation_len`) :
  - l'encodeur les lit ; le dernier état caché de sa seconde couche est le résumé de toute l'équation ;
  - ce résumé est répété 6 fois (`max_result_len`), une fois par caractère du résultat, et le décodeur lit cette séquence ;
  - la couche `nn.Linear`, appliquée à la sortie du décodeur à chaque pas, donne les logits de forme `(batch, 6, 16)`.

Ce décodeur ne reçoit pas le caractère qu'il vient d'écrire : chaque pas reçoit le même résumé de l'équation, sans *teacher forcing*. C'est le plus simple des décodeurs : il suffit à l'addition et à la soustraction, pas à la multiplication ni à la division (partie 5).

**Documentation utile**

- [`torch.nn.GRU`](https://docs.pytorch.org/docs/stable/generated/torch.nn.GRU.html)
- [`torch.Tensor.repeat`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.repeat.html)

In [ ]:
max_equation_len, max_result_len = X.shape[1], y.shape[1]


class Seq2Seq(nn.Module):
  def __init__(self, hidden_size: int = 512) -> None:
    super().__init__()
    # Votre code ici

  def forward(self, equations: torch.Tensor) -> torch.Tensor:
    # Votre code ici
    ...

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `nn.Embedding(len(index_to_char), 16)`
- `nn.GRU(16, hidden_size, num_layers=2, batch_first=True)` : la taille des vecteurs d'entrée d'abord ; `batch_first=True` pour des tenseurs de forme `(batch, longueur, …)`
- un `nn.GRU` renvoie deux valeurs : sa sortie à chaque pas, puis `h_n`, le dernier état caché de chaque couche, de forme `(couches, batch, hidden_size)` ; `h_n[-1]`, celui de la dernière couche, est de forme `(batch, hidden_size)`
- `summary.unsqueeze(1).repeat(1, max_result_len, 1)` : de `(batch, hidden_size)` à `(batch, 6, hidden_size)`, le résumé répété 6 fois
- le décodeur : `nn.GRU(hidden_size, hidden_size, batch_first=True)`, dont on garde la sortie à chaque pas, de forme `(batch, 6, hidden_size)`
- `nn.Linear(hidden_size, len(index_to_char))` agit sur la dernière dimension d'un tenseur, donc sur chaque pas séparément : appliquée à la sortie du décodeur, elle donne la forme `(batch, 6, 16)`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
class Seq2Seq(nn.Module):
  def __init__(self, hidden_size: int = 512) -> None:
    super().__init__()
    self.embedding = nn.Embedding(len(index_to_char), ...)
    self.encoder = nn.GRU(..., hidden_size, num_layers=..., batch_first=True)
    self.decoder = nn.GRU(..., hidden_size, batch_first=True)
    self.head = nn.Linear(hidden_size, ...)

  def forward(self, equations: torch.Tensor) -> torch.Tensor:
    _, hidden = self.encoder(self.embedding(equations))
    summary = hidden[...]
    repeated = summary.unsqueeze(1).repeat(1, ..., 1)
    decoded, _ = self.decoder(repeated)
    return self.head(...)
```

### Solution

In [ ]:
class Seq2Seq(nn.Module):
  """Encode the equation, then decode the result character by character."""

  def __init__(self, hidden_size: int = 512) -> None:
    super().__init__()
    # Encodeur : deux couches de GRU empilées
    self.embedding = nn.Embedding(len(index_to_char), 16)
    self.encoder = nn.GRU(16, hidden_size, num_layers=2, batch_first=True)
    # Décodeur : un GRU, puis un score par caractère à chaque pas
    self.decoder = nn.GRU(hidden_size, hidden_size, batch_first=True)
    self.head = nn.Linear(hidden_size, len(index_to_char))

  def forward(self, equations: torch.Tensor) -> torch.Tensor:
    _, hidden = self.encoder(self.embedding(equations))
    # Dernier état caché de la dernière couche : le résumé de l'équation,
    # répété pour chacun des caractères du résultat
    summary = hidden[-1]
    repeated = summary.unsqueeze(1).repeat(1, max_result_len, 1)
    decoded, _ = self.decoder(repeated)
    # Appliquée à chaque pas : (batch, 6, 16)
    return self.head(decoded)

La cellule suivante enveloppe un `Seq2Seq` dans un `SequencePredictor`, un `LightningModule` : la perte d'entropie croisée et la justesse, calculées sur chacun des 6 caractères du résultat, l'optimiseur Adam. `predict` applique un modèle à des équations, batch par batch. La cellule construit ensuite le modèle du TP et en affiche le résumé.

In [ ]:
class SequencePredictor(lightning.LightningModule):
  """Lightning wrapper: one class is predicted for each output character."""

  def __init__(self, model: nn.Module, learning_rate: float = 1e-3) -> None:
    super().__init__()
    self.save_hyperparameters(ignore=["model"])
    self.model = model
    # Une entrée d'exemple : le résumé du modèle affiche les formes des sorties
    self.example_input_array = torch.zeros(1, max_equation_len, dtype=torch.long)
    # torchmetrics demande une instance de métrique par étape
    self.accuracies = nn.ModuleDict(
      {
        f"{stage}_accuracy": torchmetrics.Accuracy(
          task="multiclass", num_classes=len(index_to_char)
        )
        for stage in ("train", "val")
      }
    )

  def forward(self, equations: torch.Tensor) -> torch.Tensor:
    return self.model(equations)

  def _step(self, batch: tuple[torch.Tensor, torch.Tensor], stage: str) -> torch.Tensor:
    equations, results = batch
    # Logits de forme (batch, 6, 16) : la perte et la métrique attendent la
    # dimension des classes en deuxième position
    logits = self(equations).transpose(1, 2)
    loss = nn.functional.cross_entropy(logits, results)
    accuracy = self.accuracies[f"{stage}_accuracy"]
    accuracy(logits, results)
    self.log(f"{stage}_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
    self.log(f"{stage}_accuracy", accuracy, on_step=False, on_epoch=True, prog_bar=True)
    return loss

  def training_step(
    self, batch: tuple[torch.Tensor, torch.Tensor], batch_index: int
  ) -> torch.Tensor:
    return self._step(batch, "train")

  def validation_step(
    self, batch: tuple[torch.Tensor, torch.Tensor], batch_index: int
  ) -> torch.Tensor:
    return self._step(batch, "val")

  def configure_optimizers(self) -> torch.optim.Optimizer:
    return torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)


@torch.no_grad()
def predict(model: nn.Module, X: torch.Tensor, batch_size: int = 1024) -> torch.Tensor:
  """Most probable character at each step, batch by batch."""
  model.eval()
  device = next(model.parameters()).device
  return torch.cat(
    [model(batch.to(device)).argmax(dim=-1).cpu() for batch in X.split(batch_size)]
  )


model = SequencePredictor(Seq2Seq())
print(ModelSummary(model, max_depth=2))

## 4. Entraîner, puis évaluer le modèle

La fonction `train` entraîne un modèle avec Lightning, sur des batchs de 512 équations, et renvoie ses métriques de chaque epoch, lues dans le journal du `CSVLogger`. Les équations de test servent à la validation.

Pendant l'entraînement, le callback `PrintExamples` affiche toutes les 10 epochs les résultats que le modèle donne pour les 20 mêmes équations de test : on le voit progresser. L'entraînement dure 30 epochs : au-delà, la perte de validation ne baisse plus, alors que celle d'entraînement baisse encore.

La justesse que Lightning affiche est celle des caractères : un résultat dont un seul caractère est faux y compte pour 5/6.

In [ ]:
def train(
  model: SequencePredictor,
  data: tuple[torch.Tensor, ...],
  name: str,
  max_epochs: int,
  callbacks: Sequence[Callback] = (),
) -> pandas.DataFrame:
  X_train, X_val, y_train, y_val = data
  trainer = lightning.Trainer(
    max_epochs=max_epochs,
    accelerator="auto",
    devices=1,
    logger=CSVLogger("logs", name=name),
    # La barre de progression de tqdm : celle de rich, que Lightning choisit
    # quand rich est installé (Colab), ne laisse pas un callback écrire avec print
    callbacks=[TQDMProgressBar(), *callbacks],
    enable_checkpointing=False,
    enable_model_summary=False,
  )
  trainer.fit(
    model,
    DataLoader(TensorDataset(X_train, y_train), batch_size=512, shuffle=True),
    DataLoader(TensorDataset(X_val, y_val), batch_size=2048),
  )
  # Métriques de chaque epoch, écrites par le CSVLogger sur des lignes
  # différentes pour l'entraînement et la validation : on les regroupe
  return (
    pandas.read_csv(pathlib.Path(trainer.logger.log_dir) / "metrics.csv")
    .groupby("epoch")
    .mean(numeric_only=True)
  )


X_examples, y_examples = X_test[:20], y_test[:20]


def examples(model: nn.Module) -> None:
  guesses = predict(model, X_examples)
  for equation, result, guess in zip(X_examples, y_examples, guesses):
    print(f"{decode(equation.flip(0))} = {decode(result)} ?= {decode(guess)}")


class PrintExamples(Callback):
  def __init__(self, every: int = 10) -> None:
    super().__init__()
    self.every = every

  def on_train_epoch_end(
    self, trainer: lightning.Trainer, pl_module: lightning.LightningModule
  ) -> None:
    epoch = trainer.current_epoch + 1
    if epoch % self.every == 0:
      print(f"\nAprès {epoch} epochs :")
      examples(pl_module)  # le modèle que le Trainer entraîne
      pl_module.train()  # predict l'a mis en mode évaluation


EPOCHS = 30
history = train(
  model,
  (X_train, X_test, y_train, y_test),
  "seq2seq",
  max_epochs=EPOCHS,
  callbacks=[PrintExamples(every=10)],
)

La fonction `evaluate` mesure, sur les équations de test :

- la proportion de prédictions illégales, qui ne se lisent pas comme un nombre (`1.2.3`) ;
- la justesse des résultats entiers : la proportion de résultats exacts, au caractère près ;
- l'erreur relative $\frac{y_{pred} - y}{y}$ des prédictions légales, dont elle trace la distribution.

In [ ]:
def evaluate(model: nn.Module) -> None:
  predictions = predict(model, X_test)
  exact = (predictions == y_test).all(dim=1)

  # Les prédictions qui se lisent comme un nombre, et leurs vrais résultats
  y_pred, y_true = [], []
  for prediction, result in zip(predictions, y_test):
    try:
      y_pred.append(float(decode(prediction)))
    except ValueError:
      continue
    y_true.append(float(decode(result)))
  y_pred, y_true = np.array(y_pred), np.array(y_true)
  illegal = 1 - len(y_pred) / len(y_test)
  print(f"Prédictions illégales : {illegal:.1%}")
  print(f"Résultats exacts : {exact.float().mean():.1%}")

  # Erreur relative, divisée par 1 quand le vrai résultat est nul
  relative_error = (y_pred - y_true) / np.where(y_true == 0, 1, y_true)
  print(f"Erreur relative absolue médiane : {np.median(np.abs(relative_error)):.2%}")

  fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
  # Des classes régulières sur l'axe logarithmique ; un résultat exact compte à 1e-4
  errors = np.maximum(np.abs(relative_error), 1e-4)
  ax1.hist(errors, bins=np.geomspace(1e-4, errors.max(), 50))
  ax1.set_xscale("log")
  ax1.set_yscale("log")
  ax1.set_title("Toutes les erreurs relatives, en valeur absolue")
  ax2.hist(relative_error, bins=20, range=(-0.1, 0.1))
  ax2.set_title("Zoom autour de 0")
  for ax in (ax1, ax2):
    ax.set_xlabel("Erreur relative")
    ax.set_ylabel("Équations")
  plt.show()


evaluate(model)

## 5. La réussite par opération

Le modèle réussit-il aussi bien les 4 opérations ? Calculez, pour chacune, la proportion d'équations de test dont le résultat prédit est **entièrement** juste, et expliquez les écarts.

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `predict(model, X_test)` : le caractère le plus probable à chaque pas, de forme `(20000, 6)`
- `(predictions == y_test).all(dim=1)` : vrai pour un résultat dont les 6 caractères sont justes
- `decode(equation.flip(0))` remet une équation dans l'ordre, et `"+" in equation` dit si c'est une addition
- un tenseur booléen sélectionne des lignes : `exact[mask]` ; `.float().mean()` donne la proportion de `True`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
predictions = predict(model, X_test)
exact = (predictions == ...).all(dim=...)
equations = [decode(equation.flip(0)) for equation in X_test]
for operation in operations:
  mask = torch.tensor([operation in equation for equation in ...])
  print(f"{operation} : {exact[...].float().mean():.1%}")
```

### Solution

In [ ]:
predictions = predict(model, X_test)
# Un résultat n'est juste que si ses 6 caractères le sont
exact = (predictions == y_test).all(dim=1)
equations = [decode(equation.flip(0)) for equation in X_test]
for operation in operations:
  mask = torch.tensor([operation in equation for equation in equations])
  print(f"{operation} : {exact[mask].float().mean():.1%} de résultats exacts")

Les écarts sont énormes : l'addition et la soustraction sont presque toujours justes (94 % de résultats exacts), la multiplication et la division presque jamais (moins de 1 %). Dans une addition ou une soustraction, chaque chiffre du résultat ne dépend que de quelques chiffres des opérandes et d'une retenue. Dans une multiplication, chaque chiffre du résultat dépend de tous ceux des opérandes, et une division donne souvent des décimales sans structure simple : ces deux opérations sont bien plus difficiles.

Le modèle n'y est pourtant pas perdu : il trouve souvent l'ordre de grandeur et les premiers chiffres (dans notre exécution, `721/866 = 0.8325` est prédit `0.8322`, et `97*727 = 70519`, `70319`), rarement tous. Le zoom de la partie 4 le montre : en dehors des résultats exacts, la plupart des erreurs relatives restent sous 10 %.

## 6. Faut-il inverser l'équation ?

`make_maths_problem` inverse les équations par défaut : l'encodeur lit `321+654` sous la forme `456+123`. Générez un jeu de données sans inversion, entraînez le même modèle dessus pendant 30 epochs, et comparez sa courbe de justesse de validation à celle du premier modèle. Comment expliquer l'écart ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `make_maths_problem(100_000, invert=False)` : les équations dans l'ordre
- `sklearn.model_selection.train_test_split` renvoie les 4 tenseurs dans l'ordre qu'attend le paramètre `data` de `train` : `(X_train, X_test, y_train, y_test)`
- `history["val_accuracy"]` : la justesse de validation de chaque epoch du premier modèle ; `[:30]` garde ses 30 premières

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
X_raw, y_raw = make_maths_problem(100_000, invert=...)
raw_data = sklearn.model_selection.train_test_split(
  X_raw, y_raw, train_size=0.8, random_state=0
)
raw_history = train(SequencePredictor(Seq2Seq()), ..., "seq2seq_raw", max_epochs=...)

plt.plot(history["val_accuracy"][:30], label=...)
plt.plot(raw_history[...], label=...)
plt.legend()
plt.show()
```

### Solution

In [ ]:
X_raw, y_raw = make_maths_problem(100_000, invert=False)
raw_data = sklearn.model_selection.train_test_split(
  X_raw, y_raw, train_size=0.8, random_state=0
)
raw_history = train(
  SequencePredictor(Seq2Seq()), raw_data, "seq2seq_raw", max_epochs=30
)

plt.plot(history["val_accuracy"][:30], label="Équation inversée")
plt.plot(raw_history["val_accuracy"], label="Équation dans l'ordre")
plt.xlabel("Epoch (à partir de 0)")
plt.ylabel("Justesse de validation (par caractère)")
plt.legend()
plt.show()

Les deux modèles progressent au même rythme pendant les 7 premières epochs, puis celui des équations inversées prend nettement l'avantage : après 10 epochs, 63 % des caractères de validation sont justes, contre 59 % dans l'ordre ; après 15, 74 % contre 64 %. Sur les équations dans l'ordre, le modèle rattrape ensuite une partie de son retard, sans le combler : 71 % contre 76 % à la 30e epoch.

Les premiers caractères du résultat dépendent surtout des premiers chiffres des opérandes, en particulier de ceux de l'opérande de gauche. Inversée, l'équation commence par l'opérande de droite et finit par celui de gauche : l'encodeur lit ces premiers chiffres en dernier, ils sont plus présents dans son dernier état caché, qui est tout ce que reçoit le décodeur. C'est l'astuce de l'article [*Sequence to Sequence Learning with Neural Networks*](https://arxiv.org/abs/1409.3215), qui inversait les phrases sources en traduction automatique.